EL siguiente codigo aqui explicado es el correspondiente al problema obligatorio 1 haciendo uso del algoritomo de Verlet para hacer una simulación del Sistema Solar. Este codigo esta escrito en en lenguaje de programación de C++. Para este codigo en algún apartado en concreto para el de calcular el periodo de los planetas, se ha hecho uso de IA , en concreto Gemini.El prompt realizado fue "Estoy haciendo uncodigo en c++ haciend uso del algoritomo de Verlet para una simulacion del sistema solar,podrias indicarme como podria hacer el calculo de los periodos de los planetas, si es posible no uses libreria que no sean #include <cmath>
#include <iostream>
#include <iomanip>
#include <fstream>
#include <array>
#include <string>
ya que son las que estoy usando en el programa"

In [ ]:
#include <cmath>
#include <iostream>
#include <iomanip>
#include <fstream>
#include <array>
#include <string>


Incluyo las librerias que voy a usar:Funciones matematicas,ficheros , entrada y salida de datos.

In [23]:


using namespace std;

// $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
// DEFINICIÓN DE CONSTANTES (PARÁMETROS)
// $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
#define ARCHIVO_ENTRADA "condiciones_i.txt" 

#define MAX_CUERPOS 100
#define ITERACIONES 100000
#define PASO_TIEMPO 0.001

#define MASA_SOLAR 1.989e30
#define UNIDAD_ASTRONOMICA 1.496e11
#define FACTOR_TIEMPO 5022004.955


Defino las constantes físicas (como la Masa Solar y la UA) y el paso de tiempo (h). también los archivos de entrada y salida

In [24]:
// ***************************************
// ARRAYS PARA LAS PROPIEDADES FÍSICAS
//*****************************************

std::array<double, MAX_CUERPOS> masas{};
std::array<double, MAX_CUERPOS> pos_x{};
std::array<double, MAX_CUERPOS> pos_y{};

std::array<double, MAX_CUERPOS> vel_x{};
std::array<double, MAX_CUERPOS> vel_y{};

std::array<double, MAX_CUERPOS> acel_x{};
std::array<double, MAX_CUERPOS> acel_y{};

std::array<double, MAX_CUERPOS> w_x{};
std::array<double, MAX_CUERPOS> w_y{}; 

// Variables para periodos
std::array<double, MAX_CUERPOS> periodos{};        
std::array<double, MAX_CUERPOS> y_ant{};       
std::array<bool, MAX_CUERPOS> tiene_vuelta{};

// Variables individuales
int n = 0;
double t = 0.0; 
long double E = 0.0, L = 0.0;


Vectores de Estado: Creo arrays para las masas, posiciones, velocidades y aceleraciones de los planetas.
Algoritmo de Verlet: Introduzco las velocidades a mitad de paso (w) . Esto sirve para predecir el movimiento en t+h/2, lo que minimiza el error y mantiene las órbitas estables.
Detección de Órbitas:  periodos: Vector para guardar el tiempo final de cada vuelta.y_ant: Guarda la posición y anterior. Al detectar un cambio de signo en y mientras x es positiva, el programa sabe que el planeta ha vuelto al punto de inicio (semieje X positivo) y ha completado un periodo exacto.
Por ultimo inicializo la variable n para bucles, el tiempo t , la energia E y el moento angular L.

In [25]:
   // 1. LECTURA DE DATOS
    ifstream entrada(ARCHIVO_ENTRADA);
    if (!entrada) {
        cout << "Error al abrir " << ARCHIVO_ENTRADA << endl;
        return 1;
    }

    while(entrada >> pos_x[n] >> vel_y[n] >> masas[n]) {
        n++;
    }
    entrada.close();

Leo los datos de las condicones iniciales de lso planetas.Y compruebo si se abre correctamente

In [26]:
   // 2. REESCALAMIENTO 
    for (int i = 0; i < n; i++) {
        pos_x[i] *= 1e6 * 1000.0 / UNIDAD_ASTRONOMICA;
        pos_y[i] = 0.0; 
        masas[i] *= 1e24 / MASA_SOLAR;
        vel_x[i] = 0.0; 
        vel_y[i] *= 1000.0 / UNIDAD_ASTRONOMICA * FACTOR_TIEMPO;
    }

AL trabajar con datos muy grandes y pequeños al mismo tiempo, tengo que reescalar las ecuaicones del movimiento

In [27]:
// 3. ARCHIVOS Y ACELERACIÓN INICIAL
    ofstream f_salida("planets_data.dat");
    ofstream f_energia("energia_momento.txt");
    // Escribir t=0
    for (int i = 0; i < n; i++) {
        f_salida << pos_x[i] << "," << pos_y[i] << "\n";
    }
    f_salida << "\n";

   // Cálculo inicial de aceleraciones según la ley de gravitación 
    for (int i = 0; i < n; i++) {
        acel_x[i] = 0.0;
        acel_y[i] = 0.0;
        
        for (int j = 0; j < n; j++) {
            if (i != j) {
                // AQUÍ DECLARAMOS dx y dy PARA PODER USARLOS
                double dx = pos_x[i] - pos_x[j];
                double dy = pos_y[i] - pos_y[j];
                
                double r = sqrt(dx * dx + dy * dy); 
                
                acel_x[i] += -masas[j] * dx / (r * r * r);
                // Si también calculas Y aquí, sería:
                acel_y[i] += -masas[j] * dy / (r * r * r); 
            }
        }
    }

Abro los ficheros en los que voy a poner los datos resultnates para la representacion grafica.Además,pongo todos los cuerpos en las pos para t=o. El segundo bloque lo que hace por paso es: Primero escoge un planeta, inicializa la aceleracion en 0,e inicia otro bucle en el cual se calcula cuanto tiran (gravedad) el resto de planetas de alrededor y se suma,y asi con todos los planetas.Para esto se usa la formula de aceleracion g pero la distancia en el denominador va al cubo, para convertir la distancia en vector unitario. La distanacia entre los planetas se halla con pitagoras en ( float r = sqrt(pow(pos_x[i]-pos_x[j], 2) + pow(pos_y[i]-pos_y[j], 2));). Otra cosa importante el tercer bucle, tiene como condicion (i !=j),esto es para que no se incluya la aceleracion del mismo planeta, ya qque la distancia es 0 y daria lugar a una división por 0.

A continuacion voy a añadir lo que seria el bucle de interaciones para aplicar el Algoritmo de verlet, al ser un bucle no puedo partirlo en celdas, por lo que , lo he separado en partes (A,B,C,D), que ire comentando paso a paso.


In [ ]:
// 4. ALGORITMO DE VERLET 
    for (int k = 0; k < ITERACIONES; k++) {
        t += PASO_TIEMPO;
        E = 0.0; L = 0.0;

        // A) Posiciones
        for (int i = 0; i < n; i++) {
            y_ant[i] = pos_y[i];

            // Velocidad intermedia W 
            w_x[i] = vel_x[i] + (PASO_TIEMPO / 2.0) * acel_x[i];
            w_y[i] = vel_y[i] + (PASO_TIEMPO / 2.0) * acel_y[i];

            // Nueva posición r(t+h) 
            // Optimización: PASO_TIEMPO * PASO_TIEMPO es más rápido que pow()
            double medio_paso_cuadrado = (PASO_TIEMPO * PASO_TIEMPO) / 2.0; 
            pos_x[i] += PASO_TIEMPO * vel_x[i] + medio_paso_cuadrado * acel_x[i];
            pos_y[i] += PASO_TIEMPO * vel_y[i] + medio_paso_cuadrado * acel_y[i];

            // Solo guardamos 1 de cada 25 pasos
            if (k % 10 == 0) {
                f_salida << pos_x[i] << "," << pos_y[i] << "\n";
            }
        }
        
        //  El salto de línea también lo hacemos solo 1 de cada 25 veces
        if (k % 10 == 0) {
            f_salida << "\n";
        }

        // B) Aceleraciones nuevas a partir de r(t+h) 
        for (int i = 0; i < n; i++) {
            acel_x[i] = 0.0; 
            acel_y[i] = 0.0;
            
            for (int j = 0; j < n; j++) {
                if (i != j) {
                    
                    double dx = pos_x[i] - pos_x[j];
                    double dy = pos_y[i] - pos_y[j];
                    double r = sqrt(dx * dx + dy * dy); 
                    double r3 = r * r * r;
                    
                    acel_x[i] += -masas[j] * dx / r3;
                    acel_y[i] += -masas[j] * dy / r3;
                }
            }
        }

        // C) Velocidades finales v(t+h)
        for (int i = 0; i < n; i++) {
            vel_x[i] = w_x[i] + (PASO_TIEMPO / 2.0) * acel_x[i];
            vel_y[i] = w_y[i] + (PASO_TIEMPO / 2.0) * acel_y[i];

            // Periodos
            if (pos_x[i] > 0.0 && pos_y[i] * y_ant[i] < 0.0 && k > 100 && i != 0 && !tiene_vuelta[i]) {
                tiene_vuelta[i] = true;
                periodos[i] = k * PASO_TIEMPO * FACTOR_TIEMPO / 86400.0;
                // Sustituimos 'endl' por '\n' (es más rápido porque no vacía el buffer de la consola innecesariamente)
                cout << "Planeta " << i << ": Periodo de " << periodos[i] << " dias.\n"; 
            }
        }

        // D) Energía y Momento
        for (int i = 0; i < n; i++) {
            L += masas[i] * (pos_x[i] * vel_y[i] - pos_y[i] * vel_x[i]);
            
            // Energía cinética: (vx*vx + vy*vy) es mejor que pow(vx, 2) + pow(vy, 2)
            E += 0.5 * masas[i] * (vel_x[i] * vel_x[i] + vel_y[i] * vel_y[i]);
            
            for (int j = 0; j < n; j++) {
                if (i != j) {
                    double dx = pos_x[i] - pos_x[j];
                    double dy = pos_y[i] - pos_y[j];
                    double r = sqrt(dx * dx + dy * dy);
                    
                    E += -masas[i] * masas[j] / (2.0 * r);
                }
            }
        } 
        f_energia << fixed << setprecision(7) << t << "\t" << E << "\t" << L << "\n";
    }

Planeta 1: Periodo de 87.9432 dias.
Planeta 2: Periodo de 224.653 dias.
Planeta 3: Periodo de 364.967 dias.
Planeta 4: Periodo de 686.283 dias.
Planeta 5: Periodo de 4326.42 dias.


EL bucle principal :Controla la evolución temporal de la simulación. En cada iteración, incremento el tiempo total según el paso de tiempo (h) definido y reinicio los contadores de energía total (E) y momento angular (L). Esto permite recalcular estos valores desde cero en cada paso para verificar su estabilidad y asegurar que se cumplen las leyes de conservación durante toda la simulación

·Bucle A (Posiciones):En esta fase del algoritmo, almaceno la posición vertical actual en y_ant para el posterior control de periodos. A continuación, calculo las velocidades intermedias w(para elsiguiente aprtado) y actualizo las posiciones de cada cuerpo mediante un desarrollo de Taylor de segundo orden (incluyendo el término de aceleración). Finalmente, exporto las nuevas coordenadas al archivo de salida.

·Bucle B (Aceleraciones nuevas a partir de r(t+h)):una vez actualizadas las posiciones, es necesario recalcular las aceleraciones de todos los cuerpos. Este bloque implementa de nuevo la Ley de Gravitación Universal, pero utilizando las coordenadas en el instante t+h. Este recálculo es el paso intermedio fundamental del algoritmo de Verlet para poder obtener, las velocidades finales con un error mínimo

·Bucle C (Velocidades finales v(t+h)):Calculo las velocidades definitivas combinando la velocidad intermedia (w)con la aceleración recién actualizada. En este mismo bloque, implemento un detector de periodos que identifica cuándo un planeta cruza el semieje X positivo. Cuando esto ocurre, el programa calcula el tiempo transcurrido, lo convierte a días terrestres y lo muestra por pantalla, registrando así la duración de un año planetario para cada cuerpo

·BUcle D (Energia y momento):En la fase final de cada iteración, se evalúan las magnitudes conservadas del sistema físico. Se calcula el momento angular total L , la energía mecánica total E sumando la energía cinética de cada cuerpo y la energía potencial gravitatoria de todos los pares  (dividiendo entre 2 para evitar el doble conteo). 

Comprobar que la energía y el momento angular se conservan es lo que nos dice la calidad de la simulación. En el universo real, ambos valores deben mantenerse constantes en un sistema planetario aislado por las leyes fundamentales de la física. EN el código, verificar que estos valores no cambian te confirma que el algoritmo matemático usado es estable y que tu paso de tiempo es correcto. Si los valores empezaran a cambiar drásticamente a lo largo de la simulación, sabriamos que  los errores numéricos se están acumulando y que las órbitas calculadas adecuadas.